## Task 1: PromptTemplate

In [ ]:
from langchain_core.prompts import PromptTemplate

# Create the prompt template
prompt = PromptTemplate(
    template='You are a helpful AI tutor. Explain the following question in simple language: {question}',
    input_variables=['question']
)

# user input dynamically
final_prompt = prompt.invoke({'question': 'what is embedding'})
print(final_prompt)

# Test with multiple inputs
questions = [
    "What is Python?",
    "What is machine learning?",
    "What is an embedding?",
    "What is RAG in Generative AI?"
]

# Printing all prompt
for question in questions:
    formatted_prompt = prompt.invoke({'question': question})
    print(f'\n{formatted_prompt}')

## Task 2: ChatPromptTemplate & Message Templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create ChatPromptTemplate
# System message defines the AI's role and behavior.
# Human message contains the user's question with a dynamic placeholder.

chat_prompt = ChatPromptTemplate.from_messages(
    [
        ('system', 'You are a helpful AI tutor. Explain concepts in simple language with examples.'),
        ('human', 'Explain the following question: {question}')
    ]
)

messages = chat_prompt.invoke({
    "question": "What is an embedding?"
})

for message in messages.to_messages():
    print(message.type, ":", message.content)

# Comparing it with PromptTemplate
print('\nComparision with PromptTemplate')
print('\nPrompt from PromptTemplate')
print(final_prompt)
print('\nPrompt from ChatPromptTemplate')
for message in messages.to_messages():
    print(message.type, ":", message.content)

## Task 3: Pydantic Output Schema

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Define the Pydantic output schema.
class Answer(BaseModel):
    answer: str = Field(description='Answer to the user question')
    confidence: float = Field(description='Estimated confidence between 0 and 1', gt=0, lt=1)
    source: str = Field(description="Source of the answer")

# Pydantic output parser using the Answer schema.
py_parser = PydanticOutputParser(pydantic_object=Answer)

# Create a prompt template.
prompt = PromptTemplate(
    template='''You are a helpful AI tutor.
    Answer the following question accurately.
    Provide a confidence estimate between 0 and 1.
    Mention the source or basis of your answer.

    follow format instruction for your response:
    {format_instructions}
    Question: {question}
    ''',
    input_variables=['question'],
    partial_variables={
        'format_instructions': py_parser.get_format_instructions()
    }
)

# Initialize the Groq chat model.
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.8)

# Build a chain using the pipe operator.
chain = prompt | llm | py_parser

# Execute the chain with a user's question.
response = chain.invoke({
    "question": "What is an embedding in machine learning?"
})

# Print the complete Pydantic object and its type.
print(response)
print(type(response))

#  Access each field of the structured response individually.
print(f'\nAnswer: {response.answer}')
print(f'Confidence: {response.confidence}')
print(f'Source: {response.source}')


## Task 4: Validation & Error Handling

In [ ]:
from langchain_core.exceptions import OutputParserException

# Validation and error Handling using try except
try:
    # Invoke the chain with error handling.
    # Using task 3 chain
    response = chain.invoke({
        "question": "What is an embedding in machine learning?"
    })

    print("Answer:", response.answer)
    print("Confidence:", response.confidence)
    print("Source:", response.source)

except OutputParserException as e:
    print("The LLM response could not be parsed.")
    print(e)

except Exception as e:
    # Handles other failures, such as API or network errors.
    print("An unexpected error occurred:")
    print(e)

## Task 5: Simple Chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Create the prompt template
prompt = PromptTemplate(
    template="You are a helpful AI tutor. Answer this question: {question}",
    input_variables=["question"]
)

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.8)

parser = StrOutputParser()

# Create the chain
chain = prompt | llm | parser

# Invoke the chain
response = chain.invoke({
    "question": "What is machine learning?"
})

# Print the response
print(response)

## Task 6: Conditional Chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Create prompts for different question types
coding_prompt = PromptTemplate(
    template="You are a coding expert. Explain with Python examples: {question}",
    input_variables=['question']
)

general_prompt = PromptTemplate(
    template="You are a helpful AI tutor. Explain simply: {question}",
    input_variables=['question']
)

parser = StrOutputParser()

code_str = ["python", "code", "programming", "function"]

branch = RunnableBranch(
    (lambda x : any(word in x['question'].lower() for word in code_str), coding_prompt | llm | parser),
    general_prompt | llm | parser
)

response = branch.invoke({'question': 'Explain Python functions'})

print(response)

## Task 7: Parallel Chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

prompt1 = PromptTemplate(
    template='Summarize this topic in 3 sentences: {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='"Provide a practical example of: {topic}',
    input_variables=['topic']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel(
    {
        'summary': prompt1 | llm | parser,
        'example': prompt2 | llm | parser
    }
)

# Invoke all chains in parallel
response = parallel_chain.invoke({'topic': 'AI'})

# Print each result
print("Summary:\n", response["summary"])
print("\nExample:\n", response["example"])

## Task 8: Runnables Basics

In [ ]:
from langchain_core.runnables import RunnableSequence, RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

# Create a prompt template
prompt_exp = PromptTemplate(template="You are a helpful AI tutor. Explain this topic: {question}", input_variables=['question'])

prompt_eg = PromptTemplate(template="Provide example of the {explaination}", input_variables=['explaination'])

# Create an output parser
parser = StrOutputParser()

# chain to generate the explanation
chain = RunnableSequence(prompt_exp, llm, parser)

# Parallel chain
parallel = RunnableParallel(
    {
    'explaination': RunnablePassthrough(),
    'example': prompt_eg | llm | parser
    }
)

# Combine both branches
final_chain = chain | parallel

response = final_chain.invoke({'question': 'Life Cycle of ML'})

# Print both generated outputs.
print("Explanation:")
print(response["explaination"])

print("\nExample:")
print(response["example"])

## Task 9: LCEL-Based RAG Chain

In [50]:
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Create sample documents for the knowledge base
documents = [
    Document(page_content="Python is a programming language used in AI and automation."),
    Document(page_content="Machine learning enables computers to learn patterns from data."),
    Document(page_content="Deep learning uses neural networks with multiple layers."),
    Document(page_content="Embeddings represent text or other data as numerical vectors."),
    Document(page_content="RAG combines information retrieval with LLM-based generation.")
]

# Initialize the embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create a FAISS vector store
vector_store = FAISS.from_documents(documents=documents,embedding=embeddings)

# Convert the vector store into a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Convert retrieved documents into a context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the RAG prompt
prompt = PromptTemplate(
template='''
You are a helpful AI tutor.
Answer the question using only the provided context.
If the answer is not in the context, say you don't know.
Context:
{context}
Question: 
{question}
''')

parser = StrOutputParser()

retrieval_chain = RunnableParallel({
    "question": RunnablePassthrough(),
    "context": retriever | RunnableLambda(format_docs)
})

# Create the LCEL RAG chain
rag_chain = retrieval_chain | prompt | llm | parser

questions = [
    "What is Python?",
    "What are embeddings?",
    "What is RAG?",
    "What is reinforcement learning?"
]

for question in questions:
    response = rag_chain.invoke(question)

    print("\nQuestion:", question)
    print("Answer:", response)
    print("-" * 50)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3638.44it/s]



Question: What is Python?
Answer: Python is a programming language used in AI and automation.
--------------------------------------------------

Question: What are embeddings?
Answer: Embeddings are representations of text (or other data) as numerical vectors.
--------------------------------------------------

Question: What is RAG?
Answer: RAG is a technique that combines information retrieval with LLM‑based generation.
--------------------------------------------------

Question: What is reinforcement learning?
Answer: I don't know.
--------------------------------------------------


## Task 10: Observations & Insights

1. Why is structured output important?
Structured output ensures that LLM responses follow a predefined format, such as JSON or a Pydantic model. It makes responses easier to validate, process, and integrate into applications.


2. What are the advantages of LCEL over traditional chains?
LCEL provides a simple way to connect components using the pipe (|) operator. It improves readability, modularity, reusability, and supports features such as streaming, batching, asynchronous execution, and parallel processing.


3. When should we use parallel vs. conditional chains?
Parallel chains are useful when multiple independent tasks need to run simultaneously, such as generating a summary, an answer, and follow-up questions. Conditional chains are useful when execution depends on a condition, such as using retrieval for factual questions and a direct LLM response for other questions.